# TTM — GPT mới từ đầu (NEW_GPT_BASELINE) · Python 3.12 · Colab Tesla T4

**Operation:** `TTM_PM_NEW_GPT_BASELINE_END_TO_END_COLAB_T4_IMPLEMENTATION_AND_Q4_ADMISSION_ONEPACK`

Mặc định chỉ chạy `QA` (10 kiểm thử nguồn, không dùng Torch/GPU hay dữ liệu riêng). Chế độ `PROBE` thực hiện đúng 3 bước tối ưu với **tensor ngẫu nhiên tổng hợp**, không có khoa học; chế độ `PREFLIGHT` chỉ xác minh dữ liệu TRAIN/VAL người dùng cung cấp và quyền được khai báo. `TRAIN_REAL` *không tự chạy*: cần chủ sở hữu thay đổi chế độ, biên nhận probe và chứng từ `owner_q4.json` khớp đầy đủ exact source/manifest/config/probe/run_id. Notebook không xác thực được chữ ký người duyệt: mọi GO nhập vào phải được phân xử độc lập bên ngoài.

Không đọc TEST, không nạp weights Small-4M cũ, không tự mount Drive, không pip install, không tự ghi GitHub, không ghi đè output. `val_bpb` chấm toàn bộ byte UTF-8 mục tiêu (0..255), loại EOS/BOS khỏi tử số/mẫu số. Chưa có chấm độc lập A/C thì không được gọi đây là chứng cứ chất lượng khoa học.

In [ ]:
from __future__ import annotations
import hashlib, json, os, pathlib, platform, subprocess, sys, urllib.request

# Chỉ đổi một biến MODE cho từng hành động; KHÔNG thay đổi source pin.
MODE = "QA"  # QA | PROBE | PREFLIGHT | TRAIN_REAL
RUN_ID = "UNSET_NO_Q4_GO"
DATA_ROOT = "/content/TTM_NEW_GPT_DATA"
DATA_MANIFEST = "/content/TTM_NEW_GPT_DATA/data_manifest.json"
PROBE_RECEIPT = "/content/TTM_NEW_GPT_PROBE.json"
OWNER_GO_FILE = "/content/TTM_NEW_GPT_OWNER_Q4_GO.json"
TRAIN_OUTPUT = "/content/ttm_new_gpt_runs/" + RUN_ID
RUNNER_GIT_BLOB = "e8dc90632598abc026d81286b1dc8f545d5b4542"
assert platform.python_version_tuple()[:2] == ("3", "12"), "PYTHON_3_12_REQUIRED"
assert MODE in {"QA", "PROBE", "PREFLIGHT", "TRAIN_REAL"}
print("MODE:", MODE, "| Python:", platform.python_version(), "| source Git blob:", RUNNER_GIT_BLOB)


In [ ]:
def git_blob_sha1(raw: bytes) -> str:
    return hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\x00" + raw).hexdigest()

# Download đúng nội dung Git blob đã khóa; không import/cài đặt từ nhánh di động khi hash đổi.
runner_url = ("https://raw.githubusercontent.com/minhtri22/TokenToModel-Lab/"
              "implementation/ttm-pm-new-gpt-baseline-colab-t4-onepack-20261117/"
              "src/ttm_new_gpt_baseline/runner.py")
with urllib.request.urlopen(runner_url, timeout=30) as response:
    runner_bytes = response.read(300_000)
assert git_blob_sha1(runner_bytes) == RUNNER_GIT_BLOB, "RUNNER_SOURCE_DRIFT_FAIL_CLOSED"
runner_dir = pathlib.Path("/content/ttm_new_gpt_onepack")
runner_dir.mkdir(parents=True, exist_ok=True)
runner_path = runner_dir / "runner.py"
if runner_path.exists():
    assert runner_path.read_bytes() == runner_bytes, "EXISTING_RUNNER_CONFLICT"
else:
    with runner_path.open("xb") as f:
        f.write(runner_bytes)
print("SOURCE_READBACK_PASS", runner_path, len(runner_bytes), hashlib.sha256(runner_bytes).hexdigest())


In [ ]:
# Chỉ cell này điều phối tác vụ. Mọi tác vụ chạy trong subprocess có timeout.
if MODE == "QA":
    cmd = [sys.executable, str(runner_path), "qa"]
    timeout = 45
elif MODE == "PROBE":
    cmd = [sys.executable, str(runner_path), "probe", "--out", PROBE_RECEIPT]
    timeout = 150
elif MODE == "PREFLIGHT":
    cmd = [sys.executable, str(runner_path), "preflight",
           "--data-root", DATA_ROOT, "--manifest", DATA_MANIFEST]
    timeout = 60
else:
    assert MODE == "TRAIN_REAL"
    assert RUN_ID != "UNSET_NO_Q4_GO", "MISSING_PROSPECTIVE_RUN_ID"
    assert pathlib.Path(PROBE_RECEIPT).is_file(), "PROBE_RECEIPT_MISSING"
    assert pathlib.Path(OWNER_GO_FILE).is_file(), "EXACT_OWNER_Q4_GO_MISSING"
    cmd = [sys.executable, str(runner_path), "train", "--data-root", DATA_ROOT,
           "--manifest", DATA_MANIFEST, "--probe-receipt", PROBE_RECEIPT,
           "--owner-go", OWNER_GO_FILE, "--run-id", RUN_ID, "--out", TRAIN_OUTPUT]
    timeout = 1020  # hard subprocess ceiling (900s application guard + overhead)
result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, check=False)
print("EXIT_CODE:", result.returncode)
print("STDOUT:", result.stdout[-12000:])
print("STDERR:", result.stderr[-12000:])
assert result.returncode == 0, "ACTION_FAILED_NO_SUCCESS_RECEIPT"


## Bàn giao evidence và điều kiện Q4

- `QA`: chụp đầu ra `PASS_SOURCE_ONLY_10_OF_10`; đây là QA kỹ thuật, không phải kiểm toán độc lập.
- `PROBE`: tệp `/content/TTM_NEW_GPT_PROBE.json` ghi số đo T4 thật trên workload **tổng hợp**, peak VRAM và môi trường. Không được suy ra tài nguyên của corpus thật đã PASS.
- `PREFLIGHT`: yêu cầu `data_manifest.json` đúng schema và các tệp `TRAIN`/`VAL`. Không cho phép nhãn `TEST`, gia đình giao nhau, byte trùng giữa splits, dữ liệu không có nguồn quyền sử dụng, hoặc SHA sai.
- `TRAIN_REAL`: chỉ sau quyết định owner Q4 riêng biệt, cùng một exact runner blob, P02 blob, config, dữ liệu và probe SHA. Không tự tạo tệp GO từ notebook. Checkpoint mới chứa model + optimizer + sampler; `READY.json` ghi SHA256; output one-shot không ghi đè.

**Trạng thái notebook tại thời điểm bàn giao:** `SOURCE_QA_CANDIDATE`; `Q4=HOLD`; `REAL_TRAINING=NOT_EXECUTED`; `TEST=NOT_ACCESSED`.